### Imports/installs

In [122]:
import re
import numpy as np
import pandas as pd, torch
from tqdm.auto import tqdm 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, roc_auc_score, classification_report, precision_recall_curve

### Remove usernames/PII

In [79]:
USERNAME_PATTERNS = [r"@\w+", r"@\[[^\]]+\]", r"@\(.*?\)", r"@[^ \t\n\r\f\v]+"]
def remove_usernames(t):
    if not isinstance(t, str):
        return ""
    for pat in USERNAME_PATTERNS:
        t = re.sub(pat, " ", t)
    return re.sub(r"\s+", " ", t).strip()

### Change labels: text, bully, category

In [90]:
def normalize_labels_for_csv(df):
    df = df.copy()
    # make header handling robust
    df.columns = [c.strip() for c in df.columns]
    # required columns
    req = {"Text","Annotation","oh_label"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}. Found: {list(df.columns)}")

    # clean text and remove @usernames
    df["text"] = df["Text"].astype(str).map(remove_usernames)

    # Annotation: 0 or 1
    df["bully"] = pd.to_numeric(df["oh_label"], errors="coerce").fillna(0).astype(int)
    df["bully"] = df["bully"].clip(0,1)

    cats = df["Annotation"].astype(str).str.lower().str.strip()
    cats = np.where(df["bully"]==1,
                    np.where(cats.str.contains("rac"), "racism",
                             np.where(cats.str.contains("sex"), "sexism", "none")),
                    "none")
    df["category"] = cats

    return df[["text","bully","category"]]

### Training: train/val/test split, build pipelines, tune threshold

In [81]:
def split_sets(df, seed=42):
    train_val, test = train_test_split(df, test_size=0.15, random_state=seed, stratify=df["bully"])
    train, val = train_test_split(train_val, test_size=0.15/(1-0.15), random_state=seed, stratify=train_val["bully"])
    return train, val, test

In [115]:
def build_bin_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            max_features=100_000,
            sublinear_tf=True,
            strip_accents="unicode",
            lowercase=True
        )),
        ("clf", LogisticRegression(
            penalty="elasticnet",
            l1_ratio=0.15,
            C=0.5,
            class_weight="balanced",
            max_iter=4000,
            solver="saga",
            n_jobs=-1,
            random_state=42
        ))
    ])

In [116]:
def build_cat_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            max_features=100_000,
            sublinear_tf=True,
            strip_accents="unicode",
            lowercase=True
        )),
        ("clf", LogisticRegression(
            penalty="elasticnet",
            l1_ratio=0.15,
            C=0.5,
            class_weight="balanced",
            max_iter=4000,
            solver="saga",
            n_jobs=-1,
            random_state=42
        ))
    ])

In [117]:
def tune_threshold(bin_model, X_val, y_val):
    proba_pos = bin_model.predict_proba(X_val)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val, proba_pos)
    f1 = (2 * precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = int(np.nanargmax(f1[:-1]))
    best_t = float(thresholds[best_idx])
    best_f1 = float(f1[best_idx])
    return best_t, best_f1

### Model evaluation

In [106]:
def eval_report(model, X, y, title):
    p = model.predict(X)
    print(f"\n== {title} ==")
    print("Accuracy:", f"{accuracy_score(y,p):.4f}")
    print("F1 (macro):", f"{f1_score(y,p,average='macro'):.4f}")
    print(classification_report(y, p, digits=4))

In [103]:
def eval_binary(model, X, y, thr, title):
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= thr).astype(int)
    acc = accuracy_score(y, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y, preds, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(y, probs)
    except ValueError:
        auc = float("nan")
    print(f"\n== {title} ==")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}  ROC-AUC: {auc:.4f}")

### Model Prediction

In [113]:
def predict_pipeline(texts, threshold=None, backstop=True, cat_conf=0.70):
    if threshold is None:
        threshold = t_star
    probs = bin_model.predict_proba(texts)[:,1]
    bpred = (probs >= threshold).astype(int)
    cats = np.array(["none"]*len(texts), dtype=object)

    if cat_model is not None:
        idx = np.where(bpred==1)[0]
        if len(idx):
            cats[idx] = cat_model.predict([texts[i] for i in idx])

        if backstop:
            tf = cat_model.named_steps["tfidf"]
            clf = cat_model.named_steps["clf"]
            Xc = tf.transform(texts)
            cprobs = clf.predict_proba(Xc)
            chat = clf.classes_[np.argmax(cprobs, axis=1)]
            cmax  = np.max(cprobs, axis=1)
            for i in range(len(texts)):
                if bpred[i]==0 and cmax[i] >= cat_conf and chat[i] in ("racism","sexism"):
                    bpred[i] = 1
                    cats[i]  = chat[i]

    out = []
    for txt, b, c in zip(texts, bpred, cats):
        out.append((txt, int(b), str(c)))
    return out

### Run: clean relabeled data, train/evaluate model

In [121]:
# 1) load and process data
df_raw = pd.read_csv("twitter_parsed_dataset.csv")
df = normalize_labels_for_csv(df_raw)

# quick sanity
display(df.head())
print(df["bully"].value_counts(dropna=False))
print(df["category"].value_counts(dropna=False))

# 2) split
train, val, test = split_sets(df, seed=42)

# 3) train binary model (with progress bar)
bin_model = build_bin_pipeline()
tfidf_bin = bin_model.named_steps["tfidf"]
clf_bin   = bin_model.named_steps["clf"]
with tqdm(total=2, desc="Training — Binary", leave=True) as pbar:
    Xtr_bin = tfidf_bin.fit_transform(train["text"])
    pbar.update(1)
    clf_bin.fit(Xtr_bin, train["bully"])
    pbar.update(1)

t_star, f1_star = tune_threshold(bin_model, val["text"], val["bully"])
print("Chosen threshold:", t_star, "F1@t* (val):", f1_star)

# 3b) binary metrics
eval_binary(bin_model, train["text"], train["bully"], t_star, "Binary — Train")
eval_binary(bin_model, val["text"],   val["bully"],   t_star, "Binary — Validation")
eval_binary(bin_model, test["text"],  test["bully"],  t_star, "Binary — Test")

# 4) train category model only on bullying rows (racism/sexism) (with progress bar)
train_b = train[train["bully"] == 1].query("category in ['racism','sexism']")
val_b   = val[val["bully"] == 1].query("category in ['racism','sexism']")
test_b  = test[test["bully"] == 1].query("category in ['racism','sexism']")

cat_model = None
if all(len(x) > 0 for x in [train_b, val_b, test_b]) and train_b["category"].nunique() == 2:
    cat_model = build_cat_pipeline()
    tfidf_cat = cat_model.named_steps["tfidf"]
    clf_cat   = cat_model.named_steps["clf"]
    with tqdm(total=2, desc="Training — Category", leave=True) as pbar:
        Xtr_cat = tfidf_cat.fit_transform(train_b["text"])
        pbar.update(1)
        clf_cat.fit(Xtr_cat, train_b["category"])
        pbar.update(1)

    # 4b) category metrics
    eval_report(cat_model, train_b["text"], train_b["category"], "Category — Train")
    eval_report(cat_model, val_b["text"],   val_b["category"],   "Category — Validation")
    eval_report(cat_model, test_b["text"],  test_b["category"],  "Category — Test")
else:
    print("Not enough bullying rows with both racism and sexism to train a category model.")

# 5) end to end predict
sample = [
    "@user you are so dumb",
    "Have a nice day",
    "women don’t belong here",
    "that policy is terrible",
]

result = predict_pipeline(sample)
result

,text,bully,category
0,I read them in context.No change in meaning. T...,0,none
1,Now you idiots claim that people who tried to ...,0,none
2,"RT Call me sexist, but when I go to an auto pl...",1,sexism
3,"Wrong, ISIS follows the example of Mohammed an...",1,racism
4,#mkr No No No No No No,0,none


bully
0    11504
1     5347
Name: count, dtype: int64
category
none      11504
sexism     3377
racism     1970
Name: count, dtype: int64


Training — Binary:   0%|          | 0/2 [00:00<?, ?it/s]

Chosen threshold: 0.5181509424096012 F1@t* (val): 0.7347962382440142

== Binary — Train ==
Accuracy: 0.8485
Precision: 0.7749  Recall: 0.7366  F1: 0.7552  ROC-AUC: 0.9134

== Binary — Validation ==
Accuracy: 0.8327
Precision: 0.7390  Recall: 0.7307  F1: 0.7348  ROC-AUC: 0.8850

== Binary — Test ==
Accuracy: 0.8133
Precision: 0.7032  Recall: 0.7120  F1: 0.7076  ROC-AUC: 0.8652


Training — Category:   0%|          | 0/2 [00:00<?, ?it/s]


== Category — Train ==
Accuracy: 0.9516
F1 (macro): 0.9488
              precision    recall  f1-score   support

      racism     0.9134    0.9613    0.9367      1394
      sexism     0.9763    0.9459    0.9609      2349

    accuracy                         0.9516      3743
   macro avg     0.9449    0.9536    0.9488      3743
weighted avg     0.9529    0.9516    0.9519      3743


== Category — Validation ==
Accuracy: 0.9302
F1 (macro): 0.9252
              precision    recall  f1-score   support

      racism     0.8738    0.9408    0.9060       287
      sexism     0.9655    0.9243    0.9444       515

    accuracy                         0.9302       802
   macro avg     0.9197    0.9325    0.9252       802
weighted avg     0.9327    0.9302    0.9307       802


== Category — Test ==
Accuracy: 0.9352
F1 (macro): 0.9306
              precision    recall  f1-score   support

      racism     0.8860    0.9412    0.9128       289
      sexism     0.9657    0.9318    0.9484       513

[('@user you are so dumb', 1, 'racism'),
 ('Have a nice day', 0, 'none'),
 ('women don’t belong here', 0, 'none'),
 ('that policy is terrible', 0, 'none')]